In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from torchvision.transforms import transforms


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = MNIST (
    root = "./data",
    train = True,
    download = True,
    transform = transform
)

test_data = MNIST (
    root = "./data",
    train = False,
    download = True,
    transform = transform
)

train_loader = DataLoader(
    train_data,
    batch_size = 64,
    shuffle = True,
    num_workers = 0
)

test_loader = DataLoader(
    test_data,
    batch_size = 64,
    shuffle = False,
    num_workers = 0
)


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()


        self.conv_layers = nn.Sequential(
            # 1st layer
            nn.Conv2d(1, 32, kernel_size =3, padding=1, stride = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel_size, stride

            # 2nd layer
            nn.Conv2d(32, 64, kernel_size =3, padding=1, stride = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) # kernel_size, stride

        )

        self.fc_layers = nn.Sequential(
            nn.Linear(3136, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flatenning
        x = self.fc_layers(x)

        return x

In [ ]:
model = CNN()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()

        outputs = model.forward(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(train_loader)}")

epoch=1/10 & loss=0.12549792262919143
epoch=2/10 & loss=0.040168878174439544
epoch=3/10 & loss=0.02736063483778684
epoch=4/10 & loss=0.02019107393220487
epoch=5/10 & loss=0.014510112971044143
epoch=6/10 & loss=0.012426456689551212
epoch=7/10 & loss=0.010119937355568587
epoch=8/10 & loss=0.008460624972993751
epoch=9/10 & loss=0.007877082095287992
epoch=10/10 & loss=0.00514753247047622


In [ ]:
import torch
import torch.nn as nn

# Redefining the CNN class in case the notebook kernel was restarted
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, stride=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(3136, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

# Create a separate, dedicated variable for the CNN model to avoid overwriting
cnn_model = CNN()

# Save the state dictionary of the CNN
torch.save(cnn_model.state_dict(), "/content/mnist_cnn.pth")
print("Saved CNN model state dictionary to /content/mnist_cnn.pth")

Saved CNN model state dictionary to /content/mnist_cnn.pth


In [ ]:
from google.colab import files

files.download("/content/mnist_cnn.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print("Model device:", next(model.parameters()).device)
print("Selected device:", device)

Model device: cuda:0
Selected device: cuda


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

correct_labels = 0
total_labels = 0

y_true = []
y_pred = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)  # Move images to GPU/CPU
        labels = labels.to(device)  # Move labels to GPU/CPU

        outputs = model(images)     # Use model(images), not model.forward(images)
        predicted = outputs.argmax(dim=1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

print(f"Accuracy: {100 * correct_labels / total_labels:.3f}%")

from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, digits=4))
print(f"Correct predictions: {correct_labels}/{total_labels}")
print(f"Wrong predictions: {total_labels - correct_labels}/{total_labels}")


Accuracy: 98.990%
[[ 978    0    0    0    0    0    0    0    1    1]
 [   0 1133    0    0    0    1    0    1    0    0]
 [   3    1 1020    0    0    0    1    6    1    0]
 [   0    1    1 1004    0    3    0    1    0    0]
 [   2    2    0    0  963    0    4    1    2    8]
 [   2    0    0    5    0  876    3    1    0    5]
 [   7    2    0    0    1    1  947    0    0    0]
 [   0    0    4    0    0    0    0 1021    1    2]
 [   8    1    2    2    0    1    0    1  956    3]
 [   0    0    1    0    3    1    0    2    1 1001]]
              precision    recall  f1-score   support

           0     0.9780    0.9980    0.9879       980
           1     0.9939    0.9982    0.9960      1135
           2     0.9922    0.9884    0.9903      1032
           3     0.9931    0.9941    0.9936      1010
           4     0.9959    0.9807    0.9882       982
           5     0.9921    0.9821    0.9870       892
           6     0.9916    0.9885    0.9901       958
           7     0

In [ ]:

import torch

torch.save(model.state_dict(), "mnist_cnn.pth")

print("Model saved successfully!")


Model saved successfully!


In [ ]:

from google.colab import files

files.download("mnist_cnn.pth")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>